# Модель предсказания возрастной категории пользователей интернет сервисов

### Общая информация
- Заказчик проекта — крупная IT-компания, которая управляет группой популярных интернет-сервисов.
- Необходимо проанализировать цифровой след пользователей, чтобы предсказать их возрастную категорию.
- Разработанный инфструмент будет использоваться маркетологами: поможет показывать рекламу на целевую аудиторию определённого возраста и снизить риски показа рекламы для взрослых несовершеннолетним.

### Постановка задачи

- разработать модель машинного обучения, которая по данным о поведении анонимного пользователя в цифровой среде будет определять его примерный возраст

### Особенности
- ML классификация: обучением с учителем, многоклассовая классификация.
- признаки могут иметь нелинейные связи с целевыми классами
- F1-мера - основная метрика, precision и recall - вспомогательные
- Модель нужно оценить одинаково по всем классам, даже если один из них встречается редко. (макро усреднение)

## Подготовка среды, библиотек, загрузка данных

In [44]:
# импорты
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from phik import phik_matrix
import joblib


In [2]:
# воспроизводимость  вычислений
RANDOM_SEED = 42 

### Загрузка данных

In [12]:
data_paths = [
    'https://code.s3.yandex.net/datasets/ds_s13_users.csv',
    'https://code.s3.yandex.net/datasets/ds_s13_visits.csv',
    'https://code.s3.yandex.net/datasets/ads_activity.csv',
    'https://code.s3.yandex.net/datasets/surf_depth.csv',
    'https://code.s3.yandex.net/datasets/primary_device.csv',
    'https://code.s3.yandex.net/datasets/cloud_usage.csv'
]

In [23]:
def read_show_info(path: str) -> pd.DataFrame:
    """Чтение csv файла по ссылке и отображение базовой информации о датасете"""
    df = pd.read_csv(path)
    display(df.info(), df.head(5))
    return df

In [24]:
df_users = read_show_info(data_paths[0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5913 entries, 0 to 5912
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   user_id       5913 non-null   object
 1   age_category  5913 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 92.5+ KB


None

,user_id,age_category
0,f545-8c95aefe8d3e5548a689-a5b2fd39,4
1,cb48-5a0d6cde4d86ae10637e-c8ceb6ed,2
2,678b-614cd47d854b9d591db2-000b2e50,0
3,4ac0-dad169100b4a29b20818-b26ae7c5,4
4,f19b-9ac21ca973b41ecfa8c3-6a58191d,0


In [25]:
df_visits = read_show_info(data_paths[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1065745 entries, 0 to 1065744
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype 
---  ------            --------------    ----- 
 0   date              1065745 non-null  object
 1   daytime           1065745 non-null  object
 2   session_id        1065745 non-null  object
 3   user_id           1065745 non-null  object
 4   website_category  1065745 non-null  object
dtypes: object(5)
memory usage: 40.7+ MB


None

,date,daytime,session_id,user_id,website_category
0,2025-11-01,вечер,066e4e02-8c1f-45eb-a50f-178659abe698,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 17
1,2025-11-01,вечер,0bce1749-3376-439c-9a22-f8ffbba00e9a,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 19
2,2025-11-01,вечер,3445d8c4-221d-4d88-bb6a-a2939fe3c610,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 18
3,2025-11-01,вечер,3bf97286-1d91-4aaa-af4a-ed58eceb8cd2,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 20
4,2025-11-01,вечер,40e22712-3cad-410d-a9f0-13bd8f6911c0,0010-5cf8f6b38a7b6c70a021-009dbcda,Category 05


In [26]:
df_ads_activity = read_show_info(data_paths[2])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5826 entries, 0 to 5825
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   user_id       5826 non-null   object
 1   ads_activity  5826 non-null   object
dtypes: object(2)
memory usage: 91.2+ KB


None

,user_id,ads_activity
0,e318-d8e69c86b543a5fb927c-c36fb6e6,очень часто
1,35cd-a972339dec534f49332c-a8b6d383,редко
2,f7e6-3b29cf9cb7ed4bb00d8f-81534360,очень редко
3,5186-e25a37549e50f45b2b43-178eaabe,умеренно
4,febd-077f277466253ee04ef6-42656680,умеренно


In [27]:
df_surf_depth = read_show_info(data_paths[3])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5715 entries, 0 to 5714
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     5715 non-null   object
 1   surf_depth  5715 non-null   object
dtypes: object(2)
memory usage: 89.4+ KB


None

,user_id,surf_depth
0,f238-0c4c1e787cce311541b7-736925a0,поверхностно
1,9030-1b562ad80182b6dc27f1-ce811740,глубоко
2,22e0-7c6cadcc45e246b8688d-c43c9b23,поверхностно
3,9d7f-a19f10756378940a49b5-5d03e1ef,поверхностно
4,4233-bb5ae4b09827e5497094-1a4956af,глубоко


In [28]:
df_primary_device = read_show_info(data_paths[4])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5669 entries, 0 to 5668
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   user_id         5669 non-null   object
 1   primary_device  5669 non-null   object
dtypes: object(2)
memory usage: 88.7+ KB


None

,user_id,primary_device
0,d602-ec060db7597a6b8cd4e7-aa625896,смартфон
1,9204-9558455be649d4e77945-b5e25d62,ПК
2,5eea-22babd6a9474b43b9d0b-a39a4cf2,ноутбук
3,c142-0296948e8d08e417de10-2da9523c,смартфон
4,abec-bb4092da51eb2233a928-e44ba074,ПК


In [ ]:
df_cloud_usage = read_show_info(data_paths[5])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5680 entries, 0 to 5679
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   user_id      5680 non-null   object
 1   cloud_usage  5680 non-null   bool  
dtypes: bool(1), object(1)
memory usage: 50.0+ KB


None

,user_id,cloud_usage
0,a1e4-91c8a52eb855595e653f-298ce305,False
1,db9a-7b8e9e94448b7fcb19b6-4edca15f,False
2,0d55-9ad768879e9b08ca7ff9-843f76c7,True
3,4baa-43285d10a6d3cc969f2a-b21881d1,False
4,b8cd-cbb2411db005115ca64d-32700c62,False


## Исследовательский анализ данных

### Базовая информация

In [100]:
def get_base_df_info(name: str, df: pd.DataFrame):
    """Информация о датафрейме для ИАД анализа"""
    
    rows_count, cols_count = df.shape
    missing = df.isna().sum()
    missing_share = (missing / rows_count * 100).round(2) if rows_count else pd.Series(0, index=df.columns, dtype=float)
    duplicates_count = (df.duplicated().sum() / rows_count * 100).round(2)
    unique_values = df.nunique(dropna=False)

    cat_cols = df.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    overview_df = pd.DataFrame({
        'Метрика': ['Строк', 'Колонок', 'Явных дубликатов, %', 'Категориальных колонок', 'Числовых колонок'],
        'Значение': [rows_count, cols_count, duplicates_count, int(len(cat_cols)), int(len(num_cols))],
    })

    summary_df = pd.DataFrame({
        'Тип данных': df.dtypes.astype(str),
        'Пропуски': missing,
        'Доля пропусков, %': missing_share,
        'Уникальные значения': unique_values,
    }).sort_values(by=['Пропуски', 'Уникальные значения'], ascending=[False, False])

    cols_df = pd.DataFrame({
        'Категориальные колонки': pd.Series(cat_cols),
        'Числовые колонки': pd.Series(num_cols),
    })

    print(f'Общая информация о датафрейме {name}')
    display(df.head())
    display(overview_df)

    print('Сводка по признакам')
    display(summary_df)

    print('Типы колонок')
    display(cols_df)
    print('-'*50)

In [101]:
data = {
    'users': df_users,
    'ads_activity': df_ads_activity,
    'cloud_usage': df_cloud_usage,
    'surf_depth': df_surf_depth,
    'visits': df_visits,
    'primary_device': df_primary_device,
}


In [110]:
name = 'cloud_usage'
get_base_df_info(name, data[name])

Общая информация о датафрейме cloud_usage


,user_id,cloud_usage
0,a1e4-91c8a52eb855595e653f-298ce305,False
1,db9a-7b8e9e94448b7fcb19b6-4edca15f,False
2,0d55-9ad768879e9b08ca7ff9-843f76c7,True
3,4baa-43285d10a6d3cc969f2a-b21881d1,False
4,b8cd-cbb2411db005115ca64d-32700c62,False


,Метрика,Значение
0,Строк,5680.0
1,Колонок,2.0
2,"Явных дубликатов, %",0.0
3,Категориальных колонок,2.0
4,Числовых колонок,0.0


Сводка по признакам


,Тип данных,Пропуски,"Доля пропусков, %",Уникальные значения
user_id,object,0,0.0,5680
cloud_usage,bool,0,0.0,2


Типы колонок


,Категориальные колонки,Числовые колонки
0,user_id,NaN
1,cloud_usage,NaN


--------------------------------------------------


### Анализ:

Данные представлены в виде 6 таблиц: 
1. users.

    Состав:
    - user_id — уникальный идентификатор пользователя.
    - age_category — возрастная категория пользователя, этот показатель модель должна научиться предсказывать

    Анализ:
    - Строк	5913, Колонок	2
    - 1.47% явных дублей 
    - Категориальных колонок 1: user_id,
    - Числовых колонок	1: age_category
    - age_category - 5 категорий
    - пропусков нет

    Итог:
    - явных дублей мало, можно удалить

2. visits

    Состав:
    - date — дата посещения сайта.
    - daytime — анонимизированное время посещения сайта. Категории: утро, день, вечер, ночь.
    - session_id — уникальный идентификатор сессии. 
    - user_id — уникальный идентификатор пользователя.
    - website_category — анонимизированная категория сайта. 

    Анализ:
    - Строк	1065745, Колонок	5
    - 1.48% явных дублей 
    - Категориальных колонок 5: date, daytime, session_id, user_id, website_category
    - Числовых колонок	0
    - пропусков нет

    Итог:
    - явных дублей мало, можно удалить
    - date стоит преобразовать к типу datetime 

3. ads_activity

    Состав:
    - user_id — уникальный идентификатор пользователя.
    - ads_activity — характеристика CTR, выраженная одним из значений: очень редко, редко, умеренно, часто, очень часто.

    Анализ:
    - Строк	5826, Колонок	2
    - 4% явных дублей 
    - Категориальных колонок 2: user_id, ads_activity
    - Числовых колонок	0
    - пропусков нет

    Итог:
    - явных дублей мало, можно удалить

4. surf_depth

    Состав:
    - user_id — уникальный идентификатор пользователя.
    - surf_depth — категориальная переменная, характеризующая глубину перехода пользователя по сайтам во время одной сессии. Содержит категории поверхностно, средне, глубоко.

    Анализ:
    - Строк	5715, Колонок	2
    - явных дублей нет
    - Категориальных колонок 2: user_id, surf_depth
    - Числовых колонок	0
    - пропусков нет

    Итог:
    - очистка от дублей не требуется, преобразование типов тоже 

5. primary_device

    Состав:
    - user_id — уникальный идентификатор пользователя.
    - primary_device — информация о типе основного устройства пользователя для выхода в Интернет.

    Анализ:
    - Строк	5669, Колонок	2
    - явных дублей нет
    - Категориальных колонок 2: user_id, primary_device
    - Числовых колонок	0
    - пропусков нет

    Итог:
    - очистка от дублей не требуется, преобразование типов тоже 

6. cloud_usage

    Состав:
    - user_id — уникальный идентификатор пользователя.
    - cloud_usage — True означает, что пользователь обращается к облачным ресурсам типа Яндекс 360 прямо или через посещаемые сайты.

    Анализ:
    - Строк	5680, Колонок	2
    - явных дублей нет
    - Категориальных колонок 2: user_id, cloud_usage
    - Числовых колонок	0
    - пропусков нет

    Итог:
    - очистка от дублей не требуется, преобразование типов тоже 

### Общий итог:

 - данные соответствуют описанию
 - присутстуют малочисленные дубликаты, от которых можно избавиться
 - категориальные переменные стоит закодировать в предобработке: OneHot, если кол-во уникальных значений < 10, иначе Target
 - 


## Предобработка данных

## Обучение и оценка базовой модели

## Создание и отбор признаков

## Подбор гиперпараметров моделей

## Подготовка артефактов модели для внедрения

## Выводы о результатах работы